## Merge Two Point Clouds

In [ ]:
import pdal
import json

def merge_laz_files(input_files, output_filename):
    # Build the pipeline dictionary: read inputs -> merge -> write output
    pipeline_dict = []
    
    # add all input files to the pipeline as readers
    for file in input_files:
        pipeline_dict.append(file)
        
    # add the merge filter
    pipeline_dict.append({
        "type": "filters.merge"
    })
    
    # add the writer to save the merged point cloud
    pipeline_dict.append({
        "type": "writers.las",
        "filename": output_filename
    })
    
    # convert the dictionary to a JSON string for PDAL
    pipeline_json = json.dumps(pipeline_dict)
    
    # create and execute the pipeline
    print(f"Merging {len(input_files)} files into {output_filename}...")
    pipeline = pdal.Pipeline(pipeline_json)
    
    try:
        count = pipeline.execute()
        print(f"Success! Merged point cloud contains {count} points.\n")
    except RuntimeError as e:
        print(f"An error occurred: {e}")


files_2023 = ["GadValley_PtCld_2023_pt1.laz", "GadValley_PtCld_2023_pt2.laz"]

files_2025 = ["GadValley_PtCld_2025_pt1.laz", "GadValley_PtCld_2025_pt2.laz"]

# merge files
merge_laz_files(files_2023, "GadValley_PtCld_2023_merged.laz")

merge_laz_files(files_2025, "GadValley_PtCld_2025_merged.laz")

In [ ]:
import numpy as np
import laspy
import matplotlib.pyplot as plt
from m3c2 import compute_m3c2  # Importing from your saved m3c2.py file

def load_las_to_numpy(filepath): # reads a las file and extracts X, Y, Z coordinates into an Nn3 array
    las = laspy.read(filepath)
    return np.vstack((las.x, las.y, las.z)).T

# load las files
cloud_ref = load_las_to_numpy("/Users/ashutoshlamsal/Desktop/Lidar/Term Project/module3/ahk/ahk_2020_uls.laz")
cloud_cmp = load_las_to_numpy("/Users/ashutoshlamsal/Desktop/Lidar/Term Project/module3/ahk/ahk_2021_uls.laz")

# to optimize the performance we are using every 100th point as the core point
corepoints = cloud_ref[::100] 

print(f"Reference Cloud Points: {cloud_ref.shape[0]}")
print(f"Compared Cloud Points: {cloud_cmp.shape[0]}")
print(f"Core Points for Calculation: {corepoints.shape[0]}")

In [ ]:
# compute M3C2 distances, LOD95, and significance
results = compute_m3c2(reference_cloud, comparison_cloud, core_points=corepoints, normal_radius=2.0, proj_diameter=1.0, max_depth=5.0, reg_error=0.03)

# Extract your arrays for plotting
m3c2_distances = results["distances"]
directions = results["normals"]
lod95 = results["lod95"]
change_sign = results["significant"]

In [ ]:
# create the figure
fig, ax = plt.subplots(1, 1, figsize=(14, 14), subplot_kw={"projection": "3d"})

# plot the distances
d = ax.scatter(corepoints[:, 0], corepoints[:, 1], corepoints[:, 2], c=m3c2_distances, cmap="seismic_r", vmin=-5.0, vmax=5.0, s=1,)
plt.colorbar(d, format=("%.2f"), label="Distance [m]", ax=ax, shrink=0.5, pad=0.15)

# add plot elements
ax.set_xlabel("Easting [m]")
ax.set_ylabel("Northing [m]")
ax.set_aspect("equal", adjustable='box') 
ax.view_init(elev=30.0, azim=120.0)

plt.tight_layout()
plt.show()